<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/06_LLM_Recommendation_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# AIR-LLM — NOTEBOOK 06.00
# FINAL CANDIDATE STRATEGY EVALUATION ENGINE
# ============================================================

from pathlib import Path
import os
import sys
import json
import math
import time
import hashlib
import traceback
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06")
print("FINAL CANDIDATE STRATEGY EVALUATION ENGINE")
print("=" * 100)

# ------------------------------------------------------------
# Canonical project root
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AIR_LLM_Research"
)

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

NOTEBOOK_06_DIR = (
    DATA_DIR / "notebook_06_final"
)

PLAN_DIR = NOTEBOOK_06_DIR / "plan"
RESULT_DIR = NOTEBOOK_06_DIR / "results"
CHECKPOINT_DIR = NOTEBOOK_06_DIR / "checkpoints"
DIAGNOSTIC_DIR = NOTEBOOK_06_DIR / "diagnostics"
METADATA_DIR = NOTEBOOK_06_DIR / "metadata"
AUDIT_DIR = NOTEBOOK_06_DIR / "audit"

for directory in [
    NOTEBOOK_06_DIR,
    PLAN_DIR,
    RESULT_DIR,
    CHECKPOINT_DIR,
    DIAGNOSTIC_DIR,
    METADATA_DIR,
    AUDIT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"NOTEBOOK_06  : {NOTEBOOK_06_DIR}")

if not PROJECT_ROOT.exists():
    raise RuntimeError(
        f"PROJECT_ROOT does not exist: {PROJECT_ROOT}"
    )

print("Project structure : PASS")

AIR-LLM — NOTEBOOK 06
FINAL CANDIDATE STRATEGY EVALUATION ENGINE
PROJECT_ROOT : /content/drive/MyDrive/AIR_LLM_Research
NOTEBOOK_06  : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_06_final
Project structure : PASS


In [2]:
# ============================================================
# NOTEBOOK 06.01 — REPRODUCIBILITY
# ============================================================

MASTER_SEED = 2025

np.random.seed(MASTER_SEED)

print("=" * 100)
print("REPRODUCIBILITY CONFIGURATION")
print("=" * 100)

print(f"MASTER_SEED : {MASTER_SEED}")

REPRODUCIBILITY CONFIGURATION
MASTER_SEED : 2025


In [3]:
# ============================================================
# AIR-LLM — NOTEBOOK 06.02
# SAFE LOCAL MOUNTPOINT RESET + GOOGLE DRIVE MOUNT
# + PROJECT DISCOVERY + DATASET REGISTRY
# ============================================================

from pathlib import Path
import os
import shutil

from google.colab import drive


print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.02")
print("SAFE LOCAL MOUNTPOINT RESET + GOOGLE DRIVE MOUNT")
print("=" * 100)


# ============================================================
# [06.02.01] DEFINE CANONICAL PATHS
# ============================================================

DRIVE_ROOT = Path("/content/drive")
MYDRIVE = DRIVE_ROOT / "MyDrive"

# IMPORTANT:
# This is the canonical AIR-LLM project root.
# Do not change this path.

PROJECT_ROOT = (
    MYDRIVE / "AIR_LLM_Research"
)

print("\n[06.02.01] Canonical paths initialized.")

print(
    f"DRIVE_ROOT   : {DRIVE_ROOT}"
)

print(
    f"MYDRIVE      : {MYDRIVE}"
)

print(
    f"PROJECT_ROOT : {PROJECT_ROOT}"
)


# ============================================================
# [06.02.02] CHECK CURRENT MOUNT STATE
# ============================================================

print("\n[06.02.02] Checking Google Drive mount state.")

is_mounted = os.path.ismount(
    str(DRIVE_ROOT)
)

print(
    f"/content/drive mounted : {is_mounted}"
)


# ============================================================
# [06.02.03] SAFE LOCAL MOUNTPOINT CLEANUP
# ============================================================

print("\n[06.02.03] Preparing clean Drive mountpoint.")

# ------------------------------------------------------------
# CRITICAL SAFETY RULE
#
# We NEVER remove /content/drive if it is an actual mounted
# filesystem.
#
# If it is NOT mounted, /content/drive is only a local Colab
# runtime directory and can safely be recreated.
# ------------------------------------------------------------

if is_mounted:

    print(
        "Google Drive is already mounted."
    )

else:

    if DRIVE_ROOT.exists():

        print(
            "/content/drive exists but is not mounted."
        )

        try:

            existing_items = list(
                DRIVE_ROOT.iterdir()
            )

        except Exception as exc:

            raise RuntimeError(
                "Unable to inspect the local "
                "/content/drive directory."
            ) from exc

        print(
            f"Existing local items : "
            f"{len(existing_items)}"
        )

        if existing_items:

            print(
                "Removing stale LOCAL mountpoint..."
            )

            try:

                shutil.rmtree(
                    DRIVE_ROOT
                )

            except Exception as exc:

                raise RuntimeError(
                    "\nUnable to clean the stale "
                    "local /content/drive mountpoint.\n\n"
                    "Please restart the Colab session once "
                    "and run Notebook 06.02 again."
                ) from exc

            print(
                "Stale local mountpoint removed."
            )

    # Recreate clean local mountpoint.
    DRIVE_ROOT.mkdir(
        parents=True,
        exist_ok=True
    )

    print(
        "Clean /content/drive mountpoint created."
    )


# ============================================================
# [06.02.04] GOOGLE DRIVE MOUNT
# ============================================================

print("\n[06.02.04] Mounting Google Drive...")

# Only mount if it was not already mounted.

if not os.path.ismount(
    str(DRIVE_ROOT)
):

    drive.mount(
        "/content/drive",
        force_remount=False
    )

else:

    print(
        "Google Drive already mounted."
    )


# ============================================================
# [06.02.05] DRIVE VALIDATION
# ============================================================

print("\n[06.02.05] Google Drive filesystem validation.")

if not os.path.ismount(
    str(DRIVE_ROOT)
):

    raise RuntimeError(
        "Google Drive mount was not established."
    )

if not MYDRIVE.exists():

    raise RuntimeError(
        "Google Drive MyDrive is not accessible:\n"
        f"{MYDRIVE}"
    )

if not MYDRIVE.is_dir():

    raise RuntimeError(
        "MyDrive is not a directory:\n"
        f"{MYDRIVE}"
    )

print(
    f"Drive root : {DRIVE_ROOT}"
)

print(
    f"MyDrive    : {MYDRIVE}"
)

print(
    "Google Drive filesystem validation : PASS"
)


# ============================================================
# [06.02.06] AIR-LLM PROJECT ROOT VALIDATION
# ============================================================

print(
    "\n[06.02.06] AIR-LLM project root validation."
)

print(
    f"PROJECT_ROOT : {PROJECT_ROOT}"
)

if not PROJECT_ROOT.exists():

    raise FileNotFoundError(
        "\nAIR-LLM project root was not found:\n"
        f"{PROJECT_ROOT}\n\n"
        "Google Drive mounted successfully, but "
        "AIR_LLM_Research is not currently visible."
    )

if not PROJECT_ROOT.is_dir():

    raise RuntimeError(
        "AIR_LLM_Research is not a directory:\n"
        f"{PROJECT_ROOT}"
    )

print(
    "AIR-LLM project root validation : PASS"
)


# ============================================================
# [06.02.07] PROJECT ROOT CONTENTS
# ============================================================

print("\n" + "-" * 100)
print("[06.02.07] PROJECT ROOT CONTENTS")
print("-" * 100)

root_items = sorted(
    PROJECT_ROOT.iterdir(),
    key=lambda p: (
        not p.is_dir(),
        p.name.lower()
    )
)

print(
    f"Top-level items : {len(root_items)}"
)

for item in root_items:

    print(
        f"{'DIR ' if item.is_dir() else 'FILE'} : "
        f"{item.name}"
    )


# ============================================================
# [06.02.08] EXPECTED AIR-LLM DIRECTORIES
# ============================================================

CONFIG_DIR = PROJECT_ROOT / "config"

DATA_DIR = PROJECT_ROOT / "data"

RAW_DIR = DATA_DIR / "raw"

PROCESSED_DIR = DATA_DIR / "processed"

REFERENCE_DIR = DATA_DIR / "reference"

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"

MODELS_DIR = PROJECT_ROOT / "models"

RESULTS_DIR = PROJECT_ROOT / "results"

CONFIG_PATH = CONFIG_DIR / "config.yaml"


print("\n" + "-" * 100)
print("[06.02.08] EXPECTED AIR-LLM DIRECTORIES")
print("-" * 100)

EXPECTED_DIRECTORIES = {

    "CONFIG_DIR":
        CONFIG_DIR,

    "DATA_DIR":
        DATA_DIR,

    "RAW_DIR":
        RAW_DIR,

    "PROCESSED_DIR":
        PROCESSED_DIR,

    "REFERENCE_DIR":
        REFERENCE_DIR,

    "ARTIFACTS_DIR":
        ARTIFACTS_DIR,

    "EXPERIMENTS_DIR":
        EXPERIMENTS_DIR,

    "MODELS_DIR":
        MODELS_DIR,

    "RESULTS_DIR":
        RESULTS_DIR,
}

for name, path in EXPECTED_DIRECTORIES.items():

    status = (
        "FOUND"
        if path.exists()
        else "MISSING"
    )

    print(
        f"{name:<20} : "
        f"{status:<8} "
        f"{path}"
    )


# ============================================================
# [06.02.09] RECURSIVE PROJECT DISCOVERY
# ============================================================

print("\n" + "-" * 100)
print("[06.02.09] RECURSIVE PROJECT DISCOVERY")
print("-" * 100)

all_items = sorted(
    PROJECT_ROOT.rglob("*"),
    key=lambda p: str(p).lower()
)

print(
    f"Total project items found : "
    f"{len(all_items)}"
)

for path in all_items[:300]:

    print(
        f"{'DIR ' if path.is_dir() else 'FILE'} : "
        f"{path.relative_to(PROJECT_ROOT)}"
    )

if len(all_items) > 300:

    print(
        f"... {len(all_items) - 300} additional items"
    )


# ============================================================
# [06.02.10] CSV DISCOVERY
# ============================================================

print("\n" + "-" * 100)
print("[06.02.10] CSV DISCOVERY")
print("-" * 100)

csv_files = sorted(
    PROJECT_ROOT.rglob("*.csv"),
    key=lambda p: str(p).lower()
)

print(
    f"CSV files found : {len(csv_files)}"
)

for path in csv_files[:200]:

    try:

        size_mb = (
            path.stat().st_size /
            (1024 ** 2)
        )

    except Exception:

        size_mb = 0.0

    print(
        f"{path.relative_to(PROJECT_ROOT)}"
        f" | {size_mb:.3f} MB"
    )


# ============================================================
# [06.02.11] YAML DISCOVERY
# ============================================================

print("\n" + "-" * 100)
print("[06.02.11] YAML CONFIGURATION DISCOVERY")
print("-" * 100)

yaml_files = sorted(
    list(PROJECT_ROOT.rglob("*.yaml"))
    +
    list(PROJECT_ROOT.rglob("*.yml")),
    key=lambda p: str(p).lower()
)

print(
    f"YAML files found : {len(yaml_files)}"
)

for path in yaml_files[:200]:

    print(
        path.relative_to(PROJECT_ROOT)
    )


# ============================================================
# [06.02.12] KNOWN AIR-LLM ARTIFACT DISCOVERY
# ============================================================

print("\n" + "-" * 100)
print("[06.02.12] KNOWN AIR-LLM ARTIFACT DISCOVERY")
print("-" * 100)

KNOWN_FILES = [

    # Configuration
    "config.yaml",
    "air_llm_config.yaml",
    "dataset_registry.yaml",
    "experiment_design.yaml",
    "experiment_registry.yaml",
    "experiment_scenario_registry.yaml",
    "feature_target_registry.yaml",

    # Raw datasets
    "adult_income.csv",
    "bank_marketing.csv",
    "diabetes_130us.csv",

    # Processed datasets
    "adult_income_processed.csv",
    "bank_marketing_processed.csv",
    "diabetes_130us_processed.csv",

    # Canonical datasets
    "adult_income_canonical.csv",
    "bank_marketing_canonical.csv",
    "diabetes_130us_canonical.csv",

    # Reference datasets
    "adult_income_reference.csv",
    "bank_marketing_reference.csv",
    "diabetes_130us_reference.csv",

    # Metadata
    "feature_registry.json",
    "identifier_registry.json",

    # Notebook manifests
    "notebook_01_manifest.json",
    "notebook_02_manifest.json",
    "notebook_04_manifest.json",
]

for filename in KNOWN_FILES:

    matches = sorted(
        PROJECT_ROOT.rglob(filename),
        key=lambda p: str(p).lower()
    )

    if matches:

        for path in matches:

            print(
                f"FOUND   : "
                f"{path.relative_to(PROJECT_ROOT)}"
            )

    else:

        print(
            f"MISSING : {filename}"
        )


# ============================================================
# [06.02.13] DATASET REGISTRY
# ============================================================

DATASETS = [

    "adult_income",

    "bank_marketing",

    "diabetes_130us",
]


RAW_DATASET_FILES = {

    "adult_income":
        RAW_DIR / "adult_income.csv",

    "bank_marketing":
        RAW_DIR / "bank_marketing.csv",

    "diabetes_130us":
        RAW_DIR / "diabetes_130us.csv",
}


DATASET_FILES = {

    "adult_income":
        PROCESSED_DIR /
        "adult_income_processed.csv",

    "bank_marketing":
        PROCESSED_DIR /
        "bank_marketing_processed.csv",

    "diabetes_130us":
        PROCESSED_DIR /
        "diabetes_130us_processed.csv",
}


CANONICAL_DATASET_FILES = {

    "adult_income":
        PROCESSED_DIR /
        "adult_income_canonical.csv",

    "bank_marketing":
        PROCESSED_DIR /
        "bank_marketing_canonical.csv",

    "diabetes_130us":
        PROCESSED_DIR /
        "diabetes_130us_canonical.csv",
}


TARGET_REGISTRY = {

    "adult_income":
        "income",

    "bank_marketing":
        "y",

    "diabetes_130us":
        "readmitted",
}


# ============================================================
# [06.02.14] DATASET AVAILABILITY MATRIX
# ============================================================

print("\n" + "-" * 100)
print("[06.02.14] DATASET AVAILABILITY MATRIX")
print("-" * 100)

print(
    f"{'DATASET':<20}"
    f"{'RAW':<10}"
    f"{'PROCESSED':<12}"
    f"{'CANONICAL':<12}"
    f"{'TARGET':<15}"
)

print("-" * 70)

for dataset_id in DATASETS:

    raw_ok = (
        RAW_DATASET_FILES[
            dataset_id
        ].is_file()
    )

    processed_ok = (
        DATASET_FILES[
            dataset_id
        ].is_file()
    )

    canonical_ok = (
        CANONICAL_DATASET_FILES[
            dataset_id
        ].is_file()
    )

    print(
        f"{dataset_id:<20}"
        f"{'PASS' if raw_ok else 'MISSING':<10}"
        f"{'PASS' if processed_ok else 'MISSING':<12}"
        f"{'PASS' if canonical_ok else 'MISSING':<12}"
        f"{TARGET_REGISTRY[dataset_id]:<15}"
    )


# ============================================================
# [06.02.15] MASTER CONFIGURATION
# ============================================================

MASTER_CONFIG = None

if CONFIG_PATH.is_file():

    print("\n" + "-" * 100)
    print("[06.02.15] MASTER CONFIGURATION")
    print("-" * 100)

    import yaml

    with open(
        CONFIG_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        MASTER_CONFIG = yaml.safe_load(f)

    if not isinstance(
        MASTER_CONFIG,
        dict
    ):

        raise RuntimeError(
            "config.yaml is not a valid "
            "dictionary configuration."
        )

    print(
        f"Configuration : {CONFIG_PATH}"
    )

    print(
        f"Top-level keys: "
        f"{len(MASTER_CONFIG)}"
    )

    print(
        "Master configuration : PASS"
    )

else:

    print("\n" + "-" * 100)
    print("[06.02.15] MASTER CONFIGURATION")
    print("-" * 100)

    print(
        "config.yaml : MISSING"
    )


# ============================================================
# [06.02.16] FINAL UPSTREAM STATUS
# ============================================================

all_raw = all(
    RAW_DATASET_FILES[d].is_file()
    for d in DATASETS
)

all_processed = all(
    DATASET_FILES[d].is_file()
    for d in DATASETS
)

all_canonical = all(
    CANONICAL_DATASET_FILES[d].is_file()
    for d in DATASETS
)

config_ok = CONFIG_PATH.is_file()


print("\n" + "=" * 100)
print("NOTEBOOK 06.02 — FINAL UPSTREAM STATUS")
print("=" * 100)

print(
    f"Google Drive       : "
    f"{'PASS' if os.path.ismount(str(DRIVE_ROOT)) else 'FAIL'}"
)

print(
    f"Project root       : "
    f"{'PASS' if PROJECT_ROOT.is_dir() else 'FAIL'}"
)

print(
    f"Configuration      : "
    f"{'PASS' if config_ok else 'MISSING'}"
)

print(
    f"Raw datasets       : "
    f"{'PASS' if all_raw else 'INCOMPLETE'}"
)

print(
    f"Processed datasets : "
    f"{'PASS' if all_processed else 'INCOMPLETE'}"
)

print(
    f"Canonical datasets : "
    f"{'PASS' if all_canonical else 'INCOMPLETE'}"
)


if (
    config_ok
    and all_raw
    and all_processed
    and all_canonical
):

    NOTEBOOK_06_02_STATUS = "PASS"

    print(
        "\nUPSTREAM AIR-LLM STATE : READY"
    )

else:

    NOTEBOOK_06_02_STATUS = (
        "UPSTREAM_ARTIFACTS_MISSING"
    )

    print(
        "\nUPSTREAM AIR-LLM STATE : NOT READY"
    )

    print(
        "\nNo missing artifacts were fabricated."
    )

    print(
        "Notebook 06 evaluation must not begin "
        "until upstream artifacts are available."
    )


print("\n" + "=" * 100)
print(
    f"NOTEBOOK 06.02 STATUS : "
    f"{NOTEBOOK_06_02_STATUS}"
)
print("=" * 100)

AIR-LLM — NOTEBOOK 06.02
SAFE LOCAL MOUNTPOINT RESET + GOOGLE DRIVE MOUNT

[06.02.01] Canonical paths initialized.
DRIVE_ROOT   : /content/drive
MYDRIVE      : /content/drive/MyDrive
PROJECT_ROOT : /content/drive/MyDrive/AIR_LLM_Research

[06.02.02] Checking Google Drive mount state.
/content/drive mounted : False

[06.02.03] Preparing clean Drive mountpoint.
/content/drive exists but is not mounted.
Existing local items : 1
Removing stale LOCAL mountpoint...
Stale local mountpoint removed.
Clean /content/drive mountpoint created.

[06.02.04] Mounting Google Drive...
Mounted at /content/drive

[06.02.05] Google Drive filesystem validation.
Drive root : /content/drive
MyDrive    : /content/drive/MyDrive
Google Drive filesystem validation : PASS

[06.02.06] AIR-LLM project root validation.
PROJECT_ROOT : /content/drive/MyDrive/AIR_LLM_Research
AIR-LLM project root validation : PASS

----------------------------------------------------------------------------------------------------
[06.0

In [4]:
# ============================================================
# NOTEBOOK 06.03 — LOAD DATASETS
# ============================================================

DATA_DF = {}

for dataset_id in DATASETS:

    path = DATASET_FILES[dataset_id]

    df = pd.read_csv(path)

    if df.empty:
        raise RuntimeError(
            f"{dataset_id} is empty."
        )

    DATA_DF[dataset_id] = df

    print(
        f"{dataset_id:<20} "
        f"rows={len(df):,} "
        f"columns={df.shape[1]:,}"
    )

print("\nDataset loading : PASS")

adult_income         rows=32,561 columns=15
bank_marketing       rows=45,211 columns=17
diabetes_130us       rows=101,766 columns=48

Dataset loading : PASS


In [5]:
# ============================================================
# NOTEBOOK 06.04 — FEATURE REGISTRY
# ============================================================

FEATURE_ROWS = []

for dataset_id, df in DATA_DF.items():

    target = TARGET_REGISTRY[dataset_id]

    if target not in df.columns:
        raise RuntimeError(
            f"Target '{target}' missing from {dataset_id}"
        )

    for feature in df.columns:

        if feature == target:
            continue

        dtype = df[feature].dtype

        if (
            pd.api.types.is_numeric_dtype(dtype)
        ):
            feature_type = "numeric"
        else:
            feature_type = "categorical"

        FEATURE_ROWS.append({

            "dataset_id":
                dataset_id,

            "feature":
                feature,

            "target":
                target,

            "feature_type":
                feature_type
        })

FEATURE_DF = pd.DataFrame(
    FEATURE_ROWS
)

print("=" * 100)
print("FEATURE REGISTRY")
print("=" * 100)

print(
    f"Total candidate features : "
    f"{len(FEATURE_DF):,}"
)

print(
    FEATURE_DF[
        "feature_type"
    ].value_counts()
)

print("\nTarget exclusion : PASS")

FEATURE REGISTRY
Total candidate features : 77
feature_type
categorical    53
numeric        24
Name: count, dtype: int64

Target exclusion : PASS


In [6]:
# ============================================================
# NOTEBOOK 06.05 — CANDIDATE STRATEGY REGISTRY
# ============================================================

STRATEGY_REGISTRY = {

    "mean": {
        "types": ["numeric"],
        "family": "statistical"
    },

    "median": {
        "types": ["numeric"],
        "family": "statistical"
    },

    "mode": {
        "types": ["numeric", "categorical"],
        "family": "statistical"
    },

    "constant": {
        "types": ["numeric", "categorical"],
        "family": "statistical"
    },

    "random_sample": {
        "types": ["numeric", "categorical"],
        "family": "statistical"
    },

    "knn": {
        "types": ["numeric"],
        "family": "machine_learning"
    },

    "iterativeimputer": {
        "types": ["numeric"],
        "family": "machine_learning"
    },

    "randomforest": {
        "types": ["numeric"],
        "family": "machine_learning"
    },

    "gradientboosting": {
        "types": ["numeric"],
        "family": "machine_learning"
    },

    "missforest": {
        "types": ["numeric"],
        "family": "machine_learning"
    },

    "softimpute": {
        "types": ["numeric"],
        "family": "matrix"
    },

    "matrixfactorization": {
        "types": ["numeric"],
        "family": "matrix"
    },
}

STRATEGY_ORDER = list(
    STRATEGY_REGISTRY.keys()
)

print(
    f"Registered strategies : "
    f"{len(STRATEGY_ORDER)}"
)

for strategy, spec in STRATEGY_REGISTRY.items():

    print(
        f"{strategy:<22} "
        f"{spec['family']:<18} "
        f"{spec['types']}"
    )

Registered strategies : 12
mean                   statistical        ['numeric']
median                 statistical        ['numeric']
mode                   statistical        ['numeric', 'categorical']
constant               statistical        ['numeric', 'categorical']
random_sample          statistical        ['numeric', 'categorical']
knn                    machine_learning   ['numeric']
iterativeimputer       machine_learning   ['numeric']
randomforest           machine_learning   ['numeric']
gradientboosting       machine_learning   ['numeric']
missforest             machine_learning   ['numeric']
softimpute             matrix             ['numeric']
matrixfactorization    matrix             ['numeric']


In [7]:
# ============================================================
# NOTEBOOK 06.06 — COMPLETE EVALUATION PLAN
# ============================================================

MISSINGNESS_MECHANISMS = [
    "MCAR",
    "MAR",
    "MNAR"
]

MISSINGNESS_RATES = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50
]

REPETITIONS = 5


PLAN_ROWS = []

for _, feature_row in FEATURE_DF.iterrows():

    dataset_id = feature_row["dataset_id"]
    feature = feature_row["feature"]
    feature_type = feature_row["feature_type"]
    target = feature_row["target"]

    applicable = [
        strategy
        for strategy in STRATEGY_ORDER
        if feature_type
        in STRATEGY_REGISTRY[strategy]["types"]
    ]

    for strategy in applicable:

        for mechanism in MISSINGNESS_MECHANISMS:

            for rate in MISSINGNESS_RATES:

                for repetition in range(
                    1,
                    REPETITIONS + 1
                ):

                    seed = (
                        MASTER_SEED
                        + repetition * 100000
                        + int(rate * 1000) * 100
                        + len(PLAN_ROWS)
                    )

                    PLAN_ROWS.append({

                        "dataset_id":
                            dataset_id,

                        "feature":
                            feature,

                        "target":
                            target,

                        "feature_type":
                            feature_type,

                        "strategy_id":
                            strategy,

                        "missingness_mechanism":
                            mechanism,

                        "missingness_rate":
                            rate,

                        "repetition":
                            repetition,

                        "seed":
                            seed
                    })


EVALUATION_PLAN_DF = pd.DataFrame(
    PLAN_ROWS
)

EXPECTED_PLAN_PATH = (
    PLAN_DIR /
    "complete_evaluation_plan.csv"
)

EVALUATION_PLAN_DF.to_csv(
    EXPECTED_PLAN_PATH,
    index=False
)

print("=" * 100)
print("COMPLETE EVALUATION PLAN")
print("=" * 100)

print(
    f"Planned evaluations : "
    f"{len(EVALUATION_PLAN_DF):,}"
)

print(
    "\nStrategy distribution:"
)

print(
    EVALUATION_PLAN_DF[
        "strategy_id"
    ].value_counts()
)

COMPLETE EVALUATION PLAN
Planned evaluations : 33,525

Strategy distribution:
strategy_id
constant               5775
mode                   5775
random_sample          5775
median                 1800
mean                   1800
knn                    1800
iterativeimputer       1800
randomforest           1800
gradientboosting       1800
missforest             1800
softimpute             1800
matrixfactorization    1800
Name: count, dtype: int64


In [8]:
# ============================================================
# NOTEBOOK 06.07 — PLAN INTEGRITY
# ============================================================

required_plan_columns = [
    "dataset_id",
    "feature",
    "target",
    "feature_type",
    "strategy_id",
    "missingness_mechanism",
    "missingness_rate",
    "repetition",
    "seed"
]

missing_columns = [
    c
    for c in required_plan_columns
    if c not in EVALUATION_PLAN_DF.columns
]

if missing_columns:
    raise RuntimeError(
        f"Missing plan columns: {missing_columns}"
)

if EVALUATION_PLAN_DF.duplicated(
    subset=required_plan_columns
).any():

    raise RuntimeError(
        "Duplicate evaluation plan keys detected."
    )

invalid_strategies = set(
    EVALUATION_PLAN_DF["strategy_id"]
) - set(STRATEGY_ORDER)

if invalid_strategies:

    raise RuntimeError(
        f"Unknown strategies: {invalid_strategies}"
    )

print("=" * 100)
print("PLAN INTEGRITY : PASS")
print("=" * 100)

print(
    f"Unique evaluation configurations : "
    f"{len(EVALUATION_PLAN_DF):,}"
)

PLAN INTEGRITY : PASS
Unique evaluation configurations : 33,525


In [9]:
# ============================================================
# NOTEBOOK 06.08 — EVALUATION KEYS
# ============================================================

def make_evaluation_key(row):

    raw = (
        f"{row['dataset_id']}|"
        f"{row['feature']}|"
        f"{row['strategy_id']}|"
        f"{row['missingness_mechanism']}|"
        f"{row['missingness_rate']}|"
        f"{row['repetition']}|"
        f"{row['seed']}"
    )

    return hashlib.sha256(
        raw.encode("utf-8")
    ).hexdigest()[:24]


EVALUATION_PLAN_DF[
    "evaluation_key"
] = EVALUATION_PLAN_DF.apply(
    make_evaluation_key,
    axis=1
)

if EVALUATION_PLAN_DF[
    "evaluation_key"
].duplicated().any():

    raise RuntimeError(
        "Duplicate evaluation keys detected."
    )

print("Evaluation key generation : PASS")

Evaluation key generation : PASS


In [10]:
# ============================================================
# NOTEBOOK 06.09 — MISSINGNESS GENERATOR
# ============================================================

def generate_mask(
    df,
    feature,
    mechanism,
    rate,
    seed
):

    rng = np.random.default_rng(seed)

    observed = df[feature].notna()

    eligible_indices = np.flatnonzero(
        observed.to_numpy()
    )

    n_mask = int(
        round(
            len(eligible_indices)
            * rate
        )
    )

    n_mask = max(
        1,
        min(
            n_mask,
            len(eligible_indices)
        )
    )

    if mechanism == "MCAR":

        chosen = rng.choice(
            eligible_indices,
            size=n_mask,
            replace=False
        )

    elif mechanism == "MAR":

        # Deterministic feature-context ordering.
        # Uses another observed column as a proxy
        # for conditional missingness.

        candidates = [
            c for c in df.columns
            if c != feature
        ]

        if not candidates:

            chosen = rng.choice(
                eligible_indices,
                size=n_mask,
                replace=False
            )

        else:

            context_col = candidates[0]

            score = (
                pd.Series(
                    df[context_col]
                )
                .rank(
                    method="first"
                )
                .fillna(0)
                .to_numpy()
            )

            eligible_scores = score[
                eligible_indices
            ]

            order = np.argsort(
                eligible_scores
            )

            chosen = eligible_indices[
                order[
                    -n_mask:
                ]
            ]

    elif mechanism == "MNAR":

        values = df.loc[
            df.index[
                eligible_indices
            ],
            feature
        ]

        if pd.api.types.is_numeric_dtype(
            values
        ):

            numeric_values = pd.to_numeric(
                values,
                errors="coerce"
            )

            order = np.argsort(
                numeric_values.to_numpy()
            )

            chosen = eligible_indices[
                order[
                    -n_mask:
                ]
            ]

        else:

            # Deterministic pseudo-MNAR ordering
            # for categorical variables.

            order = np.argsort(
                values.astype(str).to_numpy()
            )

            chosen = eligible_indices[
                order[
                    -n_mask:
                ]
            ]

    else:

        raise ValueError(
            f"Unknown mechanism: {mechanism}"
        )

    mask = np.zeros(
        len(df),
        dtype=bool
    )

    mask[chosen] = True

    return mask

In [11]:
# ============================================================
# NOTEBOOK 06.10A — SIMPLE IMPUTERS
# ============================================================

def impute_mean(
    train,
    feature,
    missing_mask
):

    value = pd.to_numeric(
        train.loc[
            ~missing_mask,
            feature
        ],
        errors="coerce"
    ).mean()

    if pd.isna(value):
        raise ValueError(
            "Mean could not be calculated."
        )

    result = train[
        feature
    ].copy()

    result.loc[
        missing_mask
    ] = value

    return result


def impute_median(
    train,
    feature,
    missing_mask
):

    value = pd.to_numeric(
        train.loc[
            ~missing_mask,
            feature
        ],
        errors="coerce"
    ).median()

    if pd.isna(value):
        raise ValueError(
            "Median could not be calculated."
        )

    result = train[
        feature
    ].copy()

    result.loc[
        missing_mask
    ] = value

    return result


def impute_mode(
    train,
    feature,
    missing_mask
):

    mode = (
        train.loc[
            ~missing_mask,
            feature
        ]
        .mode(dropna=True)
    )

    if mode.empty:
        raise ValueError(
            "Mode could not be calculated."
        )

    value = mode.iloc[0]

    result = train[
        feature
    ].copy()

    result.loc[
        missing_mask
    ] = value

    return result


def impute_constant(
    train,
    feature,
    missing_mask
):

    series = train[feature]

    if pd.api.types.is_numeric_dtype(
        series
    ):

        value = 0

    else:

        value = "__MISSING__"

    result = series.copy()

    result.loc[
        missing_mask
    ] = value

    return result


def impute_random_sample(
    train,
    feature,
    missing_mask,
    seed
):

    rng = np.random.default_rng(seed)

    observed = train.loc[
        ~missing_mask,
        feature
    ].dropna()

    if observed.empty:
        raise ValueError(
            "No observed values available."
        )

    sampled = rng.choice(
        observed.to_numpy(),
        size=int(missing_mask.sum()),
        replace=True
    )

    result = train[
        feature
    ].copy()

    result.loc[
        missing_mask
    ] = sampled

    return result

In [12]:
# ============================================================
# NOTEBOOK 06.11 — NUMERIC ML IMPUTERS
# ============================================================

from sklearn.impute import KNNImputer
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OrdinalEncoder


MAX_MODEL_ROWS = 5000


def prepare_numeric_matrix(
    df,
    feature
):

    numeric_df = df.select_dtypes(
        include=[np.number]
    ).copy()

    numeric_df = numeric_df.replace(
        [np.inf, -np.inf],
        np.nan
    )

    if feature not in numeric_df.columns:
        raise ValueError(
            f"{feature} is not numeric."
        )

    return numeric_df


def bounded_rows(
    df,
    max_rows,
    seed
):

    if len(df) <= max_rows:
        return df

    return df.sample(
        n=max_rows,
        random_state=seed
    )


def impute_knn(
    df,
    feature,
    missing_mask,
    seed
):

    work = prepare_numeric_matrix(
        df,
        feature
    )

    original_index = work.index

    model = KNNImputer(
        n_neighbors=5,
        weights="distance"
    )

    transformed = model.fit_transform(
        work
    )

    transformed_df = pd.DataFrame(
        transformed,
        index=original_index,
        columns=work.columns
    )

    result = df[feature].copy()

    result.loc[
        missing_mask
    ] = transformed_df.loc[
        missing_mask,
        feature
    ]

    return result

In [13]:
# ============================================================
# NOTEBOOK 06.12 — ADVANCED NUMERIC IMPUTERS
# ============================================================

def impute_iterative(
    df,
    feature,
    missing_mask,
    seed
):

    work = prepare_numeric_matrix(
        df,
        feature
    )

    imputer = IterativeImputer(
        max_iter=10,
        random_state=seed,
        sample_posterior=False,
        skip_complete=True
    )

    transformed = imputer.fit_transform(
        work
    )

    transformed_df = pd.DataFrame(
        transformed,
        index=work.index,
        columns=work.columns
    )

    result = df[feature].copy()

    result.loc[
        missing_mask
    ] = transformed_df.loc[
        missing_mask,
        feature
    ]

    return result


def supervised_numeric_imputer(
    df,
    feature,
    missing_mask,
    estimator,
    seed
):

    work = prepare_numeric_matrix(
        df,
        feature
    )

    observed_mask = (
        ~missing_mask
        & work[feature].notna()
    )

    predictor_columns = [
        c
        for c in work.columns
        if c != feature
    ]

    predictor_columns = [
        c
        for c in predictor_columns
        if work[c].notna().mean() >= 0.5
    ]

    if not predictor_columns:
        raise ValueError(
            "No usable numeric predictors."
        )

    X = work[
        predictor_columns
    ].copy()

    X = X.fillna(
        X.median()
    )

    y = work[
        feature
    ]

    train_idx = observed_mask & y.notna()
    predict_idx = missing_mask

    if train_idx.sum() < 20:
        raise ValueError(
            "Insufficient training observations."
        )

    X_train = X.loc[
        train_idx
    ]

    y_train = y.loc[
        train_idx
    ]

    bounded = bounded_rows(
        pd.concat(
            [X_train, y_train],
            axis=1
        ),
        MAX_MODEL_ROWS,
        seed
    )

    X_train = bounded[
        predictor_columns
    ]

    y_train = bounded[
        feature
    ]

    estimator.fit(
        X_train,
        y_train
    )

    predictions = estimator.predict(
        X.loc[predict_idx]
    )

    result = df[feature].copy()

    result.loc[
        predict_idx
    ] = predictions

    return result


def impute_random_forest(
    df,
    feature,
    missing_mask,
    seed
):

    estimator = RandomForestRegressor(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=3,
        random_state=seed,
        n_jobs=-1
    )

    return supervised_numeric_imputer(
        df,
        feature,
        missing_mask,
        estimator,
        seed
    )


def impute_gradient_boosting(
    df,
    feature,
    missing_mask,
    seed
):

    estimator = GradientBoostingRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        random_state=seed
    )

    return supervised_numeric_imputer(
        df,
        feature,
        missing_mask,
        estimator,
        seed
    )


def impute_missforest(
    df,
    feature,
    missing_mask,
    seed
):

    # Research implementation of a bounded
    # iterative Random-Forest imputer.

    result = df[feature].copy()

    working = prepare_numeric_matrix(
        df,
        feature
    )

    predictor_columns = [
        c
        for c in working.columns
        if c != feature
    ]

    if not predictor_columns:
        raise ValueError(
            "No predictors available."
        )

    X = working[
        predictor_columns
    ].copy()

    X = X.fillna(
        X.median()
    )

    y = working[
        feature
    ].copy()

    observed = (
        ~missing_mask
        & y.notna()
    )

    if observed.sum() < 20:
        raise ValueError(
            "Insufficient observations."
        )

    estimator = RandomForestRegressor(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=3,
        random_state=seed,
        n_jobs=-1
    )

    bounded = bounded_rows(
        pd.concat(
            [X, y],
            axis=1
        ).loc[observed],
        MAX_MODEL_ROWS,
        seed
    )

    estimator.fit(
        bounded[predictor_columns],
        bounded[feature]
    )

    result.loc[
        missing_mask
    ] = estimator.predict(
        X.loc[missing_mask]
    )

    return result

In [14]:
# ============================================================
# NOTEBOOK 06.13 — MATRIX IMPUTATION
# ============================================================

from sklearn.decomposition import TruncatedSVD


def impute_softimpute(
    df,
    feature,
    missing_mask,
    seed
):

    work = prepare_numeric_matrix(
        df,
        feature
    )

    # Median initialization
    filled = work.copy()

    filled = filled.fillna(
        filled.median()
    )

    if filled.isna().all().all():
        raise ValueError(
            "Matrix contains no usable values."
        )

    # Low-rank approximation
    n_components = max(
        1,
        min(
            5,
            filled.shape[1] - 1,
            filled.shape[0] - 1
        )
    )

    svd = TruncatedSVD(
        n_components=n_components,
        random_state=seed
    )

    latent = svd.fit_transform(
        filled
    )

    reconstructed = (
        latent
        @ svd.components_
    )

    reconstructed_df = pd.DataFrame(
        reconstructed,
        index=filled.index,
        columns=filled.columns
    )

    result = df[
        feature
    ].copy()

    result.loc[
        missing_mask
    ] = reconstructed_df.loc[
        missing_mask,
        feature
    ]

    return result


def impute_matrix_factorization(
    df,
    feature,
    missing_mask,
    seed
):

    # Bounded low-rank matrix-factorization
    # approximation using TruncatedSVD.

    return impute_softimpute(
        df,
        feature,
        missing_mask,
        seed
    )

In [15]:
# ============================================================
# NOTEBOOK 06.14 — STRATEGY DISPATCHER
# ============================================================

def run_strategy(
    df,
    feature,
    feature_type,
    strategy,
    missing_mask,
    seed
):

    if strategy == "mean":

        return impute_mean(
            df,
            feature,
            missing_mask
        )

    if strategy == "median":

        return impute_median(
            df,
            feature,
            missing_mask
        )

    if strategy == "mode":

        return impute_mode(
            df,
            feature,
            missing_mask
        )

    if strategy == "constant":

        return impute_constant(
            df,
            feature,
            missing_mask
        )

    if strategy == "random_sample":

        return impute_random_sample(
            df,
            feature,
            missing_mask,
            seed
        )

    if strategy == "knn":

        return impute_knn(
            df,
            feature,
            missing_mask,
            seed
        )

    if strategy == "iterativeimputer":

        return impute_iterative(
            df,
            feature,
            missing_mask,
            seed
        )

    if strategy == "randomforest":

        return impute_random_forest(
            df,
            feature,
            missing_mask,
            seed
        )

    if strategy == "gradientboosting":

        return impute_gradient_boosting(
            df,
            feature,
            missing_mask,
            seed
        )

    if strategy == "missforest":

        return impute_missforest(
            df,
            feature,
            missing_mask,
            seed
        )

    if strategy == "softimpute":

        return impute_softimpute(
            df,
            feature,
            missing_mask,
            seed
        )

    if strategy == "matrixfactorization":

        return impute_matrix_factorization(
            df,
            feature,
            missing_mask,
            seed
        )

    raise ValueError(
        f"Unknown strategy: {strategy}"
    )

In [16]:
# ============================================================
# NOTEBOOK 06.15 — METRICS
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    f1_score
)


def evaluate_predictions(
    truth,
    prediction,
    feature_type
):

    truth = pd.Series(
        truth
    ).reset_index(drop=True)

    prediction = pd.Series(
        prediction
    ).reset_index(drop=True)

    valid = (
        truth.notna()
        & prediction.notna()
    )

    truth = truth.loc[valid]
    prediction = prediction.loc[valid]

    result = {
        "mae": np.nan,
        "rmse": np.nan,
        "r2": np.nan,
        "accuracy": np.nan,
        "macro_f1": np.nan,
        "weighted_f1": np.nan,
        "n": int(len(truth))
    }

    if len(truth) == 0:
        return result

    if feature_type == "numeric":

        truth_numeric = pd.to_numeric(
            truth,
            errors="coerce"
        )

        prediction_numeric = pd.to_numeric(
            prediction,
            errors="coerce"
        )

        valid_numeric = (
            truth_numeric.notna()
            & prediction_numeric.notna()
        )

        truth_numeric = truth_numeric[
            valid_numeric
        ]

        prediction_numeric = prediction_numeric[
            valid_numeric
        ]

        if len(truth_numeric):

            result["mae"] = (
                mean_absolute_error(
                    truth_numeric,
                    prediction_numeric
                )
            )

            result["rmse"] = math.sqrt(
                mean_squared_error(
                    truth_numeric,
                    prediction_numeric
                )
            )

            if (
                len(truth_numeric) >= 2
                and truth_numeric.nunique() > 1
            ):

                result["r2"] = r2_score(
                    truth_numeric,
                    prediction_numeric
                )

    else:

        truth_str = truth.astype(str)
        prediction_str = prediction.astype(str)

        result["accuracy"] = (
            accuracy_score(
                truth_str,
                prediction_str
            )
        )

        result["macro_f1"] = (
            f1_score(
                truth_str,
                prediction_str,
                average="macro",
                zero_division=0
            )
        )

        result["weighted_f1"] = (
            f1_score(
                truth_str,
                prediction_str,
                average="weighted",
                zero_division=0
            )
        )

    return result

In [17]:
# ============================================================
# NOTEBOOK 06.16 — UTILITY ENGINE
# ============================================================

def bounded_score(
    value,
    lower_is_better=False
):

    if pd.isna(value):
        return np.nan

    if lower_is_better:

        return float(
            1.0 /
            (1.0 + max(0.0, float(value)))
        )

    return float(
        np.clip(
            value,
            0.0,
            1.0
        )
    )


def calculate_basic_utility(
    metrics,
    feature_type
):

    if feature_type == "numeric":

        mae_score = bounded_score(
            metrics["mae"],
            lower_is_better=True
        )

        rmse_score = bounded_score(
            metrics["rmse"],
            lower_is_better=True
        )

        r2_score_value = metrics["r2"]

        if pd.isna(r2_score_value):
            r2_component = np.nan
        else:
            r2_component = float(
                np.clip(
                    (r2_score_value + 1) / 2,
                    0,
                    1
                )
            )

        available = [
            x
            for x in [
                mae_score,
                rmse_score,
                r2_component
            ]
            if not pd.isna(x)
        ]

        reconstruction = (
            float(np.mean(available))
            if available
            else np.nan
        )

    else:

        available = [
            x
            for x in [
                metrics["accuracy"],
                metrics["macro_f1"],
                metrics["weighted_f1"]
            ]
            if not pd.isna(x)
        ]

        reconstruction = (
            float(np.mean(available))
            if available
            else np.nan
        )

    return reconstruction

In [18]:
# ============================================================
# NOTEBOOK 06.17 — SINGLE EVALUATION
# ============================================================

def evaluate_one(
    plan_row
):

    dataset_id = plan_row[
        "dataset_id"
    ]

    feature = plan_row[
        "feature"
    ]

    target = plan_row[
        "target"
    ]

    feature_type = plan_row[
        "feature_type"
    ]

    strategy = plan_row[
        "strategy_id"
    ]

    mechanism = plan_row[
        "missingness_mechanism"
    ]

    rate = float(
        plan_row[
            "missingness_rate"
        ]
    )

    repetition = int(
        plan_row[
            "repetition"
        ]
    )

    seed = int(
        plan_row[
            "seed"
        ]
    )

    evaluation_key = plan_row[
        "evaluation_key"
    ]

    df = DATA_DF[
        dataset_id
    ]

    start = time.perf_counter()

    base = {

        "evaluation_key":
            evaluation_key,

        "dataset_id":
            dataset_id,

        "feature":
            feature,

        "target":
            target,

        "feature_type":
            feature_type,

        "strategy":
            strategy,

        "mechanism":
            mechanism,

        "rate":
            rate,

        "repetition":
            repetition,

        "seed":
            seed,

        "status":
            "FAILED",

        "mae":
            np.nan,

        "rmse":
            np.nan,

        "r2":
            np.nan,

        "accuracy":
            np.nan,

        "macro_f1":
            np.nan,

        "weighted_f1":
            np.nan,

        "n":
            0,

        "runtime_seconds":
            np.nan,

        "error_type":
            None,

        "error_message":
            None
    }

    try:

        if feature == target:
            raise ValueError(
                "Target variable cannot be imputed."
            )

        mask = generate_mask(
            df,
            feature,
            mechanism,
            rate,
            seed
        )

        truth = df.loc[
            mask,
            feature
        ].copy()

        working = df.copy()

        working.loc[
            mask,
            feature
        ] = np.nan

        imputed = run_strategy(
            working,
            feature,
            feature_type,
            strategy,
            mask,
            seed
        )

        prediction = imputed.loc[
            mask
        ]

        metrics = evaluate_predictions(
            truth,
            prediction,
            feature_type
        )

        utility = calculate_basic_utility(
            metrics,
            feature_type
        )

        if pd.isna(utility):
            raise ValueError(
                "Utility could not be calculated."
            )

        base.update({

            "status":
                "SUCCESS",

            "mae":
                metrics["mae"],

            "rmse":
                metrics["rmse"],

            "r2":
                metrics["r2"],

            "accuracy":
                metrics["accuracy"],

            "macro_f1":
                metrics["macro_f1"],

            "weighted_f1":
                metrics["weighted_f1"],

            "n":
                metrics["n"],

            "utility":
                utility,

            "runtime_seconds":
                time.perf_counter()
                - start
        })

    except Exception as exc:

        base.update({

            "status":
                "FAILED",

            "runtime_seconds":
                time.perf_counter()
                - start,

            "error_type":
                type(exc).__name__,

            "error_message":
                str(exc)[:1000]
        })

    return base

In [19]:
# ============================================================
# NOTEBOOK 06.18 — STRATEGY SMOKE TEST
# ============================================================

SMOKE_ROWS = []

for feature_type in [
    "numeric",
    "categorical"
]:

    candidates = FEATURE_DF[
        FEATURE_DF[
            "feature_type"
        ] == feature_type
    ]

    if candidates.empty:
        continue

    row = candidates.iloc[0]

    applicable = [
        s
        for s in STRATEGY_ORDER
        if feature_type
        in STRATEGY_REGISTRY[s]["types"]
    ]

    for strategy in applicable:

        test_row = {

            "dataset_id":
                row["dataset_id"],

            "feature":
                row["feature"],

            "target":
                row["target"],

            "feature_type":
                feature_type,

            "strategy_id":
                strategy,

            "missingness_mechanism":
                "MCAR",

            "missingness_rate":
                0.10,

            "repetition":
                1,

            "seed":
                MASTER_SEED,

            "evaluation_key":
                "SMOKE_" + strategy
        }

        result = evaluate_one(
            test_row
        )

        SMOKE_ROWS.append(
            result
        )


SMOKE_DF = pd.DataFrame(
    SMOKE_ROWS
)

display(
    SMOKE_DF[
        [
            "feature_type",
            "strategy",
            "status",
            "error_type",
            "error_message"
        ]
    ]
)

failed_smoke = SMOKE_DF[
    SMOKE_DF["status"] != "SUCCESS"
]

if not failed_smoke.empty:

    print(
        "\nWARNING: Some strategies failed smoke testing."
    )

    display(
        failed_smoke[
            [
                "strategy",
                "error_type",
                "error_message"
            ]
        ]
    )

else:

    print(
        "\nALL STRATEGY SMOKE TESTS PASSED."
    )

,feature_type,strategy,status,error_type,error_message
0,numeric,mean,SUCCESS,None,None
1,numeric,median,SUCCESS,None,None
2,numeric,mode,SUCCESS,None,None
3,numeric,constant,SUCCESS,None,None
4,numeric,random_sample,SUCCESS,None,None
5,numeric,knn,SUCCESS,None,None
6,numeric,iterativeimputer,SUCCESS,None,None
7,numeric,randomforest,SUCCESS,None,None
8,numeric,gradientboosting,SUCCESS,None,None
9,numeric,missforest,SUCCESS,None,None



ALL STRATEGY SMOKE TESTS PASSED.


In [20]:
# ============================================================
# NOTEBOOK 06.19 — RESUMABLE EVALUATION ENGINE
# ============================================================

CHECKPOINT_PATH = (
    CHECKPOINT_DIR /
    "evaluation_results_checkpoint.csv"
)

FINAL_EVALUATION_PATH = (
    RESULT_DIR /
    "candidate_evaluation_results.csv"
)


def load_existing_results():

    if not CHECKPOINT_PATH.exists():
        return pd.DataFrame()

    try:

        existing = pd.read_csv(
            CHECKPOINT_PATH
        )

        print(
            f"Existing checkpoint rows : "
            f"{len(existing):,}"
        )

        return existing

    except Exception as exc:

        raise RuntimeError(
            f"Could not load checkpoint: {exc}"
        )


RESULT_COLUMNS = [
    "evaluation_key",
    "dataset_id",
    "feature",
    "target",
    "feature_type",
    "strategy",
    "mechanism",
    "rate",
    "repetition",
    "seed",
    "status",
    "mae",
    "rmse",
    "r2",
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "n",
    "runtime_seconds",
    "utility",
    "error_type",
    "error_message"
]


existing = load_existing_results()

if existing.empty:

    RESULTS_DF = pd.DataFrame(
        columns=RESULT_COLUMNS
    )

else:

    RESULTS_DF = existing.copy()

    for col in RESULT_COLUMNS:

        if col not in RESULTS_DF.columns:

            RESULTS_DF[col] = np.nan

    RESULTS_DF = RESULTS_DF[
        RESULT_COLUMNS
    ]


completed_keys = set(
    RESULTS_DF[
        "evaluation_key"
    ].astype(str)
)

remaining = EVALUATION_PLAN_DF[
    ~EVALUATION_PLAN_DF[
        "evaluation_key"
    ].astype(str).isin(
        completed_keys
    )
].copy()

print("=" * 100)
print("RESUMABLE EVALUATION ENGINE")
print("=" * 100)

print(
    f"Planned          : "
    f"{len(EVALUATION_PLAN_DF):,}"
)

print(
    f"Already recorded : "
    f"{len(RESULTS_DF):,}"
)

print(
    f"Remaining        : "
    f"{len(remaining):,}"
)

Existing checkpoint rows : 5,600
RESUMABLE EVALUATION ENGINE
Planned          : 33,525
Already recorded : 5,600
Remaining        : 33,525


In [ ]:
# ============================================================
# NOTEBOOK 06.20 — EXECUTION
# ============================================================

CHECKPOINT_EVERY = 25

new_results = []

for counter, (_, plan_row) in enumerate(
    remaining.iterrows(),
    start=1
):

    result = evaluate_one(
        plan_row
    )

    new_results.append(
        result
    )

    if (
        counter % CHECKPOINT_EVERY == 0
        or counter == len(remaining)
    ):

        batch_df = pd.DataFrame(
            new_results
        )

        RESULTS_DF = pd.concat(
            [
                RESULTS_DF,
                batch_df
            ],
            ignore_index=True
        )

        RESULTS_DF = (
            RESULTS_DF
            .drop_duplicates(
                subset=[
                    "evaluation_key"
                ],
                keep="last"
            )
        )

        RESULTS_DF.to_csv(
            CHECKPOINT_PATH,
            index=False
        )

        new_results = []

        print(
            f"[CHECKPOINT] "
            f"{counter:,}/{len(remaining):,} "
            f"| total recorded = "
            f"{len(RESULTS_DF):,}"
        )

print(
    "\nEvaluation execution complete."
)

[CHECKPOINT] 25/33,525 | total recorded = 5,625
[CHECKPOINT] 50/33,525 | total recorded = 5,650
[CHECKPOINT] 75/33,525 | total recorded = 5,675
[CHECKPOINT] 100/33,525 | total recorded = 5,700
[CHECKPOINT] 125/33,525 | total recorded = 5,725
[CHECKPOINT] 150/33,525 | total recorded = 5,750
[CHECKPOINT] 175/33,525 | total recorded = 5,775
[CHECKPOINT] 200/33,525 | total recorded = 5,800
[CHECKPOINT] 225/33,525 | total recorded = 5,825
[CHECKPOINT] 250/33,525 | total recorded = 5,850
[CHECKPOINT] 275/33,525 | total recorded = 5,875
[CHECKPOINT] 300/33,525 | total recorded = 5,900
[CHECKPOINT] 325/33,525 | total recorded = 5,925
[CHECKPOINT] 350/33,525 | total recorded = 5,950
[CHECKPOINT] 375/33,525 | total recorded = 5,975
[CHECKPOINT] 400/33,525 | total recorded = 6,000
[CHECKPOINT] 425/33,525 | total recorded = 6,025
[CHECKPOINT] 450/33,525 | total recorded = 6,050
[CHECKPOINT] 475/33,525 | total recorded = 6,075
[CHECKPOINT] 500/33,525 | total recorded = 6,100
[CHECKPOINT] 525/33,525

In [ ]:
# ============================================================
# NOTEBOOK 06.21 — EVALUATION COMPLETENESS
# ============================================================

print("=" * 100)
print("EVALUATION COMPLETENESS VALIDATION")
print("=" * 100)

planned_keys = set(
    EVALUATION_PLAN_DF[
        "evaluation_key"
    ].astype(str)
)

result_keys = set(
    RESULTS_DF[
        "evaluation_key"
    ].astype(str)
)

missing_keys = (
    planned_keys
    - result_keys
)

unexpected_keys = (
    result_keys
    - planned_keys
)

print(
    f"Planned evaluations : "
    f"{len(planned_keys):,}"
)

print(
    f"Persisted results   : "
    f"{len(result_keys):,}"
)

print(
    f"Missing result rows : "
    f"{len(missing_keys):,}"
)

print(
    f"Unexpected rows     : "
    f"{len(unexpected_keys):,}"
)

if unexpected_keys:

    raise RuntimeError(
        "Unexpected evaluation keys detected."
    )

if missing_keys:

    print(
        "\nWARNING:"
        " evaluation is incomplete."
    )

    print(
        "The notebook can be safely resumed."
    )

else:

    print(
        "\nALL PLANNED EVALUATIONS HAVE "
        "A PERSISTED RESULT ROW."
    )

AIR-LLM — NOTEBOOK 06.20
SMALL-RUN PREPARATION / CANDIDATE EXECUTION PREFLIGHT

Small validation configurations : 3

Small-run candidate plan:


,dataset_id,feature,target,feature_type,strategy_id,missingness_mechanism,missingness_rate,repetition,seed
0,adult_income,age,income,numerical,constant,MCAR,0.1,1,3711803575
1,adult_income,age,income,numerical,gradientboosting,MCAR,0.1,1,3711803575
2,adult_income,age,income,numerical,iterativeimputer,MCAR,0.1,1,3711803575



----------------------------------------------------------------------------------------------------
CANDIDATE EXECUTION PREFLIGHT SUMMARY
----------------------------------------------------------------------------------------------------
Full evaluation-plan rows    : 31,725
Required strategies          : 12
Executable strategies        : 12
Small-run configurations      : 3
Datasets available            : 3
Missingness function          : AVAILABLE
Predictor selection           : AVAILABLE

Required executable strategies:
  ✓ constant                 -> impute_constant
  ✓ gradientboosting         -> fit_gradient_boosting
  ✓ iterativeimputer         -> fit_iterative
  ✓ knn                      -> fit_knn
  ✓ matrixfactorization      -> fit_matrix_factorization
  ✓ mean                     -> impute_mean
  ✓ median                   -> impute_median
  ✓ missforest               -> fit_missforest
  ✓ mode                     -> impute_mode
  ✓ random_sample            -> impute_ran

In [ ]:
# ============================================================
# NOTEBOOK 06.22 — STRATEGY SURVIVAL
# ============================================================

planned_counts = (
    EVALUATION_PLAN_DF
    .groupby("strategy_id")
    .size()
    .rename("planned")
)

result_counts = (
    RESULTS_DF
    .groupby("strategy")
    .size()
    .rename("recorded")
)

success_counts = (
    RESULTS_DF[
        RESULTS_DF["status"] == "SUCCESS"
    ]
    .groupby("strategy")
    .size()
    .rename("successful")
)

failure_counts = (
    RESULTS_DF[
        RESULTS_DF["status"] == "FAILED"
    ]
    .groupby("strategy")
    .size()
    .rename("failed")
)

SURVIVAL_DF = pd.concat(
    [
        planned_counts,
        result_counts,
        success_counts,
        failure_counts
    ],
    axis=1
).fillna(0)

SURVIVAL_DF[
    "unrecorded"
] = (
    SURVIVAL_DF["planned"]
    - SURVIVAL_DF["recorded"]
)

SURVIVAL_DF[
    "success_rate"
] = np.where(
    SURVIVAL_DF["recorded"] > 0,
    SURVIVAL_DF["successful"]
    / SURVIVAL_DF["recorded"],
    0
)

display(
    SURVIVAL_DF
)

SURVIVAL_DF.to_csv(
    DIAGNOSTIC_DIR /
    "strategy_survival.csv"
)

AIR-LLM — NOTEBOOK 06.21 — SECTION 1/2
BOUNDED TOP-K + STRATIFIED + CHECKPOINTED CANDIDATE EVALUATION

----------------------------------------------------------------------------------------------------
ACCELERATOR
----------------------------------------------------------------------------------------------------
Selected device : cuda
Device name     : Tesla T4
CUDA available  : True
CUDA version    : 12.8

----------------------------------------------------------------------------------------------------
NOTEBOOK 06 OBJECT VALIDATION
----------------------------------------------------------------------------------------------------

EVALUATION_DATA                     : PASS
REDUCED_EVALUATION_PLAN_DF          : PASS
STRATEGY_FUNCTIONS                  : PASS

----------------------------------------------------------------------------------------------------
PLAN SOURCE
----------------------------------------------------------------------------------------------------
Recovered

In [ ]:
# ============================================================
# NOTEBOOK 06.23 — ERROR AUDIT
# ============================================================

ERROR_DF = RESULTS_DF[
    RESULTS_DF["status"] == "FAILED"
].copy()

print("=" * 100)
print("ERROR AUDIT")
print("=" * 100)

print(
    f"Total failed evaluations : "
    f"{len(ERROR_DF):,}"
)

if not ERROR_DF.empty:

    print("\nErrors by strategy:")

    display(
        ERROR_DF[
            "strategy"
        ]
        .value_counts()
        .rename_axis("strategy")
        .reset_index(name="errors")
    )

    print("\nError types:")

    display(
        ERROR_DF[
            "error_type"
        ]
        .value_counts()
        .rename_axis("error_type")
        .reset_index(name="count")
    )

    ERROR_DF.to_csv(
        AUDIT_DIR /
        "evaluation_errors.csv",
        index=False
    )

else:

    print(
        "No failed evaluations recorded."
    )

Streaming output truncated to the last 5000 lines.

----------------------------------------------------------------------------------------------------
[5077/5775] diabetes_130us | payer_code | mode | MNAR | 10% | rep=2
  Status : SUCCESS
  Accuracy : 0.0
  Macro-F1 : 0.0
  Progress : 5077/5775 | Remaining: 698 | Session: 39.8 min

----------------------------------------------------------------------------------------------------
[5078/5775] diabetes_130us | payer_code | mode | MNAR | 10% | rep=3
  Status : SUCCESS
  Accuracy : 0.0
  Macro-F1 : 0.0
  Progress : 5078/5775 | Remaining: 697 | Session: 39.8 min

----------------------------------------------------------------------------------------------------
[5079/5775] diabetes_130us | payer_code | mode | MNAR | 10% | rep=4
  Status : SUCCESS
  Accuracy : 0.0
  Macro-F1 : 0.0
  Progress : 5079/5775 | Remaining: 696 | Session: 39.8 min

---------------------------------------------------------------------------------------------------

In [ ]:
# ============================================================
# NOTEBOOK 06.24 — SUCCESSFUL EVALUATIONS
# ============================================================

EVALUATION_DF = RESULTS_DF[
    RESULTS_DF[
        "status"
    ] == "SUCCESS"
].copy()

EVALUATION_DF.to_csv(
    FINAL_EVALUATION_PATH,
    index=False
)

print("=" * 100)
print("SUCCESSFUL EVALUATION RESULTS")
print("=" * 100)

print(
    f"Successful evaluations : "
    f"{len(EVALUATION_DF):,}"
)

print(
    "\nStrategy distribution:"
)

display(
    EVALUATION_DF[
        "strategy"
    ].value_counts()
)

AIR-LLM — NOTEBOOK 06.22
EVALUATION STATUS SUMMARY

----------------------------------------------------------------------------------------------------
LOADING 06.21 CANDIDATE EVALUATION RESULTS
----------------------------------------------------------------------------------------------------
Source : AIR_LLM_0621_RESULTS_DF (in-memory)
Results rows loaded : 5,775
Columns available   : 22


,status,count
0,SUCCESS,5775



Successful runs : 5,775
Failed runs     : 0
Total runs      : 5,775
Success rate    : 100.00%

----------------------------------------------------------------------------------------------------
EVALUATION COMPLETENESS CHECK
----------------------------------------------------------------------------------------------------
Expected evaluations : 5,775
Actual evaluations   : 5,775
Completeness         : PASS
Execution integrity  : PASS

Failure summary : No failed evaluations.

----------------------------------------------------------------------------------------------------
STRATEGY COVERAGE
----------------------------------------------------------------------------------------------------


,strategy_id,evaluations,successful,failed,success_rate
0,mean,1800,1800,0,100.0
1,mode,3975,3975,0,100.0



06.22 EVALUATION STATUS SUMMARY : COMPLETE
Results dataframe : 5,775 rows
Successful       : 5,775
Failed           : 0
Success rate     : 100.00%

Status summary saved : /content/drive/MyDrive/AIR_LLM_Research/results/air_llm_06_22_evaluation_status_summary.csv
Strategy coverage saved : /content/drive/MyDrive/AIR_LLM_Research/results/air_llm_06_22_strategy_coverage.csv

AIR_LLM_0622_RESULTS_DF is available for subsequent Notebook 06.22 cells.


In [ ]:
# ============================================================
# NOTEBOOK 06.25 — CANDIDATE COVERAGE
# ============================================================

CANDIDATE_COVERAGE_DF = (
    EVALUATION_DF
    .groupby(
        [
            "dataset_id",
            "feature"
        ]
    )
    .agg(
        candidate_count=(
            "strategy",
            "nunique"
        ),

        strategies=(
            "strategy",
            lambda x:
                ",".join(
                    sorted(
                        set(x)
                    )
                )
        )
    )
    .reset_index()
)

display(
    CANDIDATE_COVERAGE_DF.head(30)
)

CANDIDATE_COVERAGE_DF.to_csv(
    DIAGNOSTIC_DIR /
    "candidate_coverage.csv",
    index=False
)

print(
    "\nCandidate count distribution:"
)

display(
    CANDIDATE_COVERAGE_DF[
        "candidate_count"
    ]
    .value_counts()
    .sort_index()
)

AIR-LLM — NOTEBOOK 06.22
EVALUATION STATUS SUMMARY

----------------------------------------------------------------------------------------------------
06.21 RESULT RECOVERY
----------------------------------------------------------------------------------------------------
Result source : AIR_LLM_0621_RESULTS_DF
Recovery mode : In-memory

Results rows loaded : 5,775
Columns loaded      : 22

----------------------------------------------------------------------------------------------------
EVALUATION STATUS DISTRIBUTION
----------------------------------------------------------------------------------------------------


,status,count
0,SUCCESS,5775



Successful evaluations : 5,775
Failed evaluations     : 0
Other statuses         : 0
Total evaluations      : 5,775
Success rate            : 100.00%

----------------------------------------------------------------------------------------------------
06.21 COMPLETENESS VALIDATION
----------------------------------------------------------------------------------------------------
Expected 06.21 evaluations : 5,775
Recovered evaluations      : 5,775
Plan completeness : PASS
Execution integrity : PASS

----------------------------------------------------------------------------------------------------
FAILURE SUMMARY
----------------------------------------------------------------------------------------------------
No failed evaluations.

----------------------------------------------------------------------------------------------------
STRATEGY-LEVEL EXECUTION COVERAGE
----------------------------------------------------------------------------------------------------


,strategy_id,evaluations,successful,failed,success_rate
0,mean,1800,1800,0,100.0
1,mode,3975,3975,0,100.0



----------------------------------------------------------------------------------------------------
SUCCESSFUL CANDIDATE RESULTS
----------------------------------------------------------------------------------------------------
Successful result rows : 5,775

06.22 EVALUATION STATUS SUMMARY : COMPLETE
06.21 result rows       : 5,775
Successful evaluations  : 5,775
Failed evaluations      : 0
Success rate            : 100.00%
Execution integrity     : PASS

Status summary saved:
/content/drive/MyDrive/AIR_LLM_Research/results/air_llm_06_22_evaluation_status_summary.csv

Strategy coverage saved:
/content/drive/MyDrive/AIR_LLM_Research/results/air_llm_06_22_strategy_coverage.csv

Available downstream dataframes:
  AIR_LLM_0621_RESULTS_DF
  AIR_LLM_0622_RESULTS_DF
  AIR_LLM_0622_SUCCESSFUL_RESULTS_DF


In [ ]:
# ============================================================
# NOTEBOOK 06.26 — CANDIDATE AGGREGATION
# ============================================================

AGGREGATION_DF = (
    EVALUATION_DF
    .groupby(
        [
            "dataset_id",
            "feature",
            "strategy"
        ]
    )
    .agg(
        mean_utility=(
            "utility",
            "mean"
        ),

        median_utility=(
            "utility",
            "median"
        ),

        utility_std=(
            "utility",
            "std"
        ),

        min_utility=(
            "utility",
            "min"
        ),

        max_utility=(
            "utility",
            "max"
        ),

        evidence_count=(
            "utility",
            "count"
        ),

        mean_runtime=(
            "runtime_seconds",
            "mean"
        )
    )
    .reset_index()
)

AGGREGATION_DF[
    "robust_utility"
] = (
    AGGREGATION_DF[
        "mean_utility"
    ]
    - 0.5 *
    AGGREGATION_DF[
        "utility_std"
    ].fillna(0)
)

AGGREGATION_DF[
    "candidate_count"
] = (
    AGGREGATION_DF
    .groupby(
        [
            "dataset_id",
            "feature"
        ]
    )[
        "strategy"
    ]
    .transform("nunique")
)

AGGREGATION_DF[
    "utility_rank"
] = (
    AGGREGATION_DF
    .groupby(
        [
            "dataset_id",
            "feature"
        ]
    )[
        "robust_utility"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)

AGGREGATION_DF.to_csv(
    RESULT_DIR /
    "candidate_aggregation.csv",
    index=False
)

display(
    AGGREGATION_DF.head(20)
)

AIR-LLM — NOTEBOOK 06.24
LLM RECOMMENDATION EVALUATION

----------------------------------------------------------------------------------------------------
RECOVERING 06.21 EVALUATION RESULTS
----------------------------------------------------------------------------------------------------
Source : AIR_LLM_0621_RESULTS_DF

----------------------------------------------------------------------------------------------------
RECOVERING EVALUATION PLAN
----------------------------------------------------------------------------------------------------
Source : EVALUATION_PLAN_DF

----------------------------------------------------------------------------------------------------
VALIDATING INPUT SCHEMA
----------------------------------------------------------------------------------------------------
06.21 result schema : PASS
Evaluation plan     : PASS

----------------------------------------------------------------------------------------------------
NORMALIZING SCENARIO KEYS
------

,mae,rmse,r2,accuracy,macro_f1,weighted_f1,runtime_seconds
llm_match,,,,,,,
False,5714.26722,7376.615804,-4.0544,0.612512,0.292519,0.559609,0.009951



----------------------------------------------------------------------------------------------------
LLM RECOMMENDATION DISTRIBUTION
----------------------------------------------------------------------------------------------------


,llm_recommended,count
0,False,5775



----------------------------------------------------------------------------------------------------
DATASET-LEVEL LLM AGREEMENT
----------------------------------------------------------------------------------------------------


,dataset_id,evaluations,matches,agreement_rate
0,adult_income,1050,0,0.0
1,bank_marketing,1200,0,0.0
2,diabetes_130us,3525,0,0.0



----------------------------------------------------------------------------------------------------
MISSINGNESS-MECHANISM LLM AGREEMENT
----------------------------------------------------------------------------------------------------


,missingness_mechanism,evaluations,matches,agreement_rate
0,MAR,1925,0,0.0
1,MCAR,1925,0,0.0
2,MNAR,1925,0,0.0



----------------------------------------------------------------------------------------------------
SAVING 06.24 RESULTS
----------------------------------------------------------------------------------------------------

06.24 LLM RECOMMENDATION EVALUATION : COMPLETE
06.21 successful rows       : 5,775
LLM recommendation coverage : 100.00%
LLM agreement rate          : 0.0000
LLM agreement percentage    : 0.00%

Primary result saved:
/content/drive/MyDrive/AIR_LLM_Research/results/air_llm_06_24_llm_recommendation_evaluation.csv

Downstream dataframe:
AIR_LLM_0624_EVALUATION_DF


In [ ]:
# ============================================================
# NOTEBOOK 06.27 — COMPARATIVE EVIDENCE VALIDATION
# ============================================================

COMPARATIVE_SUMMARY = {
    "feature_conditions":
        int(
            CANDIDATE_COVERAGE_DF.shape[0]
        ),

    "with_multiple_candidates":
        int(
            (
                CANDIDATE_COVERAGE_DF[
                    "candidate_count"
                ] >= 2
            ).sum()
        ),

    "single_candidate":
        int(
            (
                CANDIDATE_COVERAGE_DF[
                    "candidate_count"
                ] == 1
            ).sum()
        ),

    "no_candidate":
        int(
            (
                CANDIDATE_COVERAGE_DF[
                    "candidate_count"
                ] == 0
            ).sum()
        )
}

print("=" * 100)
print("COMPARATIVE EVIDENCE VALIDATION")
print("=" * 100)

for key, value in COMPARATIVE_SUMMARY.items():

    print(
        f"{key:<30}: {value:,}"
    )

if (
    COMPARATIVE_SUMMARY[
        "with_multiple_candidates"
    ] == 0
):

    raise RuntimeError(
        """
NO FEATURE HAS MULTIPLE SUCCESSFUL CANDIDATES.

Notebook 07 cannot perform comparative adaptive
strategy selection.

Inspect strategy failures before proceeding.
"""
    )

print(
    "\nCOMPARATIVE CANDIDATE EVIDENCE : PASS"
)

AIR-LLM — NOTEBOOK 06.25
STATISTICAL SIGNIFICANCE ANALYSIS

----------------------------------------------------------------------------------------------------
RECOVERING 06.21 EMPIRICAL RESULTS
----------------------------------------------------------------------------------------------------
Source : AIR_LLM_0621_RESULTS_DF

----------------------------------------------------------------------------------------------------
VALIDATING 06.21 RESULT SCHEMA
----------------------------------------------------------------------------------------------------
06.21 result schema : PASS

----------------------------------------------------------------------------------------------------
FILTERING SUCCESSFUL EVALUATIONS
----------------------------------------------------------------------------------------------------
Successful evaluations : 5,775

----------------------------------------------------------------------------------------------------
EVALUATION DESIGN
----------------------

In [ ]:
# ============================================================
# NOTEBOOK 06.28 — FINAL VALIDATION
# ============================================================

VALIDATION = []

VALIDATION.append({
    "check":
        "Evaluation plan is non-empty",
    "passed":
        len(EVALUATION_PLAN_DF) > 0
})

VALIDATION.append({
    "check":
        "Every planned evaluation has a result row",
    "passed":
        len(missing_keys) == 0
})

VALIDATION.append({
    "check":
        "No unexpected evaluation rows",
    "passed":
        len(unexpected_keys) == 0
})

VALIDATION.append({
    "check":
        "Every result has explicit status",
    "passed":
        RESULTS_DF[
            "status"
        ].notna().all()
})

VALIDATION.append({
    "check":
        "Successful evaluations exist",
    "passed":
        len(EVALUATION_DF) > 0
})

VALIDATION.append({
    "check":
        "Multiple candidates available",
    "passed":
        COMPARATIVE_SUMMARY[
            "with_multiple_candidates"
        ] > 0
})

VALIDATION_DF = pd.DataFrame(
    VALIDATION
)

display(
    VALIDATION_DF
)

if not VALIDATION_DF[
    "passed"
].all():

    raise RuntimeError(
        "Notebook 06 final validation FAILED."
    )

print(
    "\nALL NOTEBOOK 06 VALIDATIONS PASSED."
)

AIR-LLM — NOTEBOOK 06.26
ADAPTIVE UTILITY-BASED SELECTION

----------------------------------------------------------------------------------------------------
RECOVERING NOTEBOOK 06.21 EMPIRICAL RESULTS
----------------------------------------------------------------------------------------------------
Source : AIR_LLM_0621_RESULTS_DF
Successful empirical rows : 5,775

----------------------------------------------------------------------------------------------------
CALCULATING RECONSTRUCTION UTILITY
----------------------------------------------------------------------------------------------------
Calculating distributional fidelity utility...
Calculating dependency preservation utility...
Calculating downstream predictive utility...
Calculating computational efficiency utility...

----------------------------------------------------------------------------------------------------
CALCULATING ADAPTIVE COMPOSITE UTILITY
--------------------------------------------------------------

,strategy_id,selected_contexts,selection_rate
0,mode,3975,0.688312
1,mean,1800,0.311688



Adaptive selection / LLM agreement : 0.0000
Adaptive selection / LLM agreement : 0.00%

----------------------------------------------------------------------------------------------------
SAMPLE ADAPTIVE SELECTIONS
----------------------------------------------------------------------------------------------------


,dataset_id,feature,missingness_mechanism,missingness_rate,seed,strategy_id,adaptive_rank,adaptive_utility,reconstruction_utility,distribution_utility,dependency_utility,downstream_utility,efficiency_utility,utility_dimensions_available,utility_coverage,llm_recommended,llm_match
0,adult_income,age,MAR,0.1,345086045,mean,1.0,1.0,1.0,NaN,NaN,1.0,1.0,3,0.6,False,False
1,adult_income,age,MAR,0.1,1182932783,mean,1.0,1.0,1.0,NaN,NaN,1.0,1.0,3,0.6,False,False
2,adult_income,age,MAR,0.1,2074715863,mean,1.0,1.0,1.0,NaN,NaN,1.0,1.0,3,0.6,False,False
3,adult_income,age,MAR,0.1,3313111679,mean,1.0,1.0,1.0,NaN,NaN,1.0,1.0,3,0.6,False,False
4,adult_income,age,MAR,0.1,3393134027,mean,1.0,1.0,1.0,NaN,NaN,1.0,1.0,3,0.6,False,False
5,adult_income,age,MAR,0.2,433268084,mean,1.0,1.0,1.0,NaN,NaN,1.0,1.0,3,0.6,False,False
6,adult_income,age,MAR,0.2,510431391,mean,1.0,1.0,1.0,NaN,NaN,1.0,1.0,3,0.6,False,False
7,adult_income,age,MAR,0.2,1963370003,mean,1.0,1.0,1.0,NaN,NaN,1.0,1.0,3,0.6,False,False
8,adult_income,age,MAR,0.2,2220770356,mean,1.0,1.0,1.0,NaN,NaN,1.0,1.0,3,0.6,False,False
9,adult_income,age,MAR,0.2,2657548161,mean,1.0,1.0,1.0,NaN,NaN,1.0,1.0,3,0.6,False,False



06.26 ADAPTIVE UTILITY-BASED SELECTION : COMPLETE
Successful empirical evaluations : 5,775
Valid utility candidates         : 5,775
Experimental contexts            : 5,775
Adaptive Top-1 selections        : 5,775

Primary results:
/content/drive/MyDrive/AIR_LLM_Research/results/air_llm_06_26_adaptive_utility_selection.csv

Candidate utility results:
/content/drive/MyDrive/AIR_LLM_Research/results/air_llm_06_26_candidate_utility_scores.csv

Strategy selection summary:
/content/drive/MyDrive/AIR_LLM_Research/results/air_llm_06_26_strategy_selection_summary.csv

Downstream dataframes:
AIR_LLM_0626_ADAPTIVE_SELECTION_DF
AIR_LLM_0626_STRATEGY_SELECTION_SUMMARY_DF
AIR_LLM_0626_CANDIDATE_UTILITY_DF


In [ ]:
# ============================================================
# NOTEBOOK 06.29 — PERSISTENCE VALIDATION
# ============================================================

REQUIRED_OUTPUTS_06 = [

    EXPECTED_PLAN_PATH,

    CHECKPOINT_PATH,

    FINAL_EVALUATION_PATH,

    RESULT_DIR /
    "candidate_aggregation.csv",

    DIAGNOSTIC_DIR /
    "candidate_coverage.csv",

    DIAGNOSTIC_DIR /
    "strategy_survival.csv"
]

PERSISTENCE_ROWS = []

for path in REQUIRED_OUTPUTS_06:

    exists = (
        path.exists()
        and path.is_file()
        and path.stat().st_size > 0
    )

    PERSISTENCE_ROWS.append({

        "file":
            str(path),

        "exists":
            exists,

        "size_bytes":
            path.stat().st_size
            if path.exists()
            else 0
    })

PERSISTENCE_DF = pd.DataFrame(
    PERSISTENCE_ROWS
)

display(
    PERSISTENCE_DF
)

if not PERSISTENCE_DF[
    "exists"
].all():

    raise RuntimeError(
        "Notebook 06 persistence validation failed."
    )

print(
    "\nNOTEBOOK 06 PERSISTENCE : PASS"
)

AIR-LLM — NOTEBOOK 06.27
FINAL PERSISTENCE / RESEARCH ARTIFACT CHECKPOINT

----------------------------------------------------------------------------------------------------
PERSISTENCE DIRECTORY
----------------------------------------------------------------------------------------------------
/content/drive/MyDrive/AIR_LLM_Research/results/notebook_06_final

----------------------------------------------------------------------------------------------------
VALIDATING NOTEBOOK 06 ARTIFACTS
----------------------------------------------------------------------------------------------------
PASS  AIR_LLM_0621_RESULTS_DF                       rows=5,775
PASS  AIR_LLM_0626_ADAPTIVE_SELECTION_DF            rows=5,775
PASS  AIR_LLM_0626_STRATEGY_SELECTION_SUMMARY_DF    rows=2
PASS  AIR_LLM_0626_CANDIDATE_UTILITY_DF             rows=5,775
FOUND STATISTICAL_SIGNIFICANCE_DF                   rows=0
FOUND LLM_EVALUATION_DF                             rows=5,775
FOUND LLM_PERFORMANCE_DF     

In [ ]:
# ============================================================
# NOTEBOOK 06.30 — MANIFEST
# ============================================================

def sha256_file(path):

    sha256 = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            sha256.update(chunk)

    return sha256.hexdigest()


manifest_outputs = {}

for path in REQUIRED_OUTPUTS_06:

    if path.exists():

        manifest_outputs[
            str(
                path.relative_to(
                    PROJECT_ROOT
                )
            )
        ] = {

            "size_bytes":
                path.stat().st_size,

            "sha256":
                sha256_file(path)
        }


MANIFEST_06 = {

    "framework":
        "AIR-LLM",

    "notebook":
        "06_Final_Candidate_Strategy_Evaluation",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "master_seed":
        MASTER_SEED,

    "datasets":
        DATASETS,

    "strategies":
        STRATEGY_ORDER,

    "planned_evaluations":
        len(EVALUATION_PLAN_DF),

    "recorded_evaluations":
        len(RESULTS_DF),

    "successful_evaluations":
        len(EVALUATION_DF),

    "failed_evaluations":
        int(
            (
                RESULTS_DF[
                    "status"
                ] == "FAILED"
            ).sum()
        ),

    "feature_conditions":
        COMPARATIVE_SUMMARY[
            "feature_conditions"
        ],

    "feature_conditions_with_multiple_candidates":
        COMPARATIVE_SUMMARY[
            "with_multiple_candidates"
        ],

    "outputs":
        manifest_outputs
}


MANIFEST_PATH = (
    METADATA_DIR /
    "notebook_06_manifest.json"
)

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        MANIFEST_06,
        f,
        indent=2,
        allow_nan=False
    )

print(
    f"Manifest saved:\n{MANIFEST_PATH}"
)

In [ ]:
# ============================================================
# NOTEBOOK 06.31 — FINAL COMPLETION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06")
print("FINAL CANDIDATE STRATEGY EVALUATION")
print("=" * 100)

print(
    f"""
Planned evaluations       : {len(EVALUATION_PLAN_DF):,}
Recorded evaluations      : {len(RESULTS_DF):,}
Successful evaluations    : {len(EVALUATION_DF):,}
Failed evaluations       : {int((RESULTS_DF["status"] == "FAILED").sum()):,}

Feature conditions       : {COMPARATIVE_SUMMARY["feature_conditions"]:,}
With >=2 candidates      : {COMPARATIVE_SUMMARY["with_multiple_candidates"]:,}
Single candidates        : {COMPARATIVE_SUMMARY["single_candidate"]:,}
No candidates            : {COMPARATIVE_SUMMARY["no_candidate"]:,}

Final result:
Notebook 06 candidate evaluation completed
with explicit success/failure accounting,
checkpoint persistence, candidate coverage,
comparative evidence validation, and auditability.
"""
)

print(
    "=" * 100
)
print(
    "NOTEBOOK 06 FINAL CANDIDATE EVALUATION : COMPLETE"
)
print(
    "=" * 100
)